# KMeans Sector CAGR Analysis - NIFTY IT

This notebook downloads 5-minute intraday candles for Indian IT stocks,
tiles the data six times to simulate a full year, computes return and risk
features for each stock, clusters them with KMeans, and produces a trade signal.

Stocks covered: TCS, Infosys, Wipro, HCL Tech, Tech Mahindra, LTIMindtree,
Persistent, Mphasis, Coforge, OFSS, Hexaware, Birlasoft, Cyient, Mastek, Zensar.

## Step 1 - Install and Import Libraries

Run this once at the start of every session.

In [ ]:
# Install dependencies (Colab resets on each session)
!pip install yfinance scikit-learn pandas numpy matplotlib seaborn --quiet

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yfinance as yf
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Apply a consistent, clean style to every plot in this notebook
plt.rcParams.update({
    'figure.facecolor'  : '#FFFFFF',
    'axes.facecolor'    : '#F8F9FA',
    'axes.edgecolor'    : '#DEE2E6',
    'axes.grid'         : True,
    'grid.color'        : '#E9ECEF',
    'grid.linewidth'    : 0.8,
    'font.family'       : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'xtick.color'       : '#495057',
    'ytick.color'       : '#495057',
    'axes.labelcolor'   : '#343A40',
    'text.color'        : '#212529',
})

print('Libraries loaded successfully.')

## Step 2 - Sector Configuration

All tunable parameters live here. Change the stock list to switch sectors.

In [ ]:
SECTOR_NAME = 'NIFTY IT'

# 15 NIFTY IT stocks - mix of large cap, mid cap, and small cap
stocks = [
    'TCS.NS',
    'INFY.NS',
    'WIPRO.NS',
    'HCLTECH.NS',
    'TECHM.NS',
    'LTIM.NS',
    'PERSISTENT.NS',
    'MPHASIS.NS',
    'COFORGE.NS',
    'OFSS.NS',
    'HEXAWARE.NS',
    'BSOFT.NS',
    'CYIENT.NS',
    'MASTEK.NS',
    'ZENSARTECH.NS',
]

# Data settings
INTERVAL        = '5m'   # 5-minute candles
PERIOD          = '60d'  # max yfinance allows for 5m
CANDLES_PER_DAY = 75     # 375 trading minutes / 5 = 75 bars
ITERATIONS      = 6      # repeat 2-month block x6 to get ~1 year

# Clustering settings
N_CLUSTERS  = 3
KMEANS_RUNS = 5          # pick the best run out of 5

print(f'Sector  : {SECTOR_NAME}')
print(f'Stocks  : {len(stocks)}')
print(f'Period  : {PERIOD} at {INTERVAL} interval')
print(f'Simulate: {ITERATIONS} iterations = approx 1 year')
print(f'Clusters: {N_CLUSTERS}')

## Step 3 - Download 5-Minute Candle Data

yfinance caps 5-minute data at 60 days. We download each stock one at a time
so a single failure does not abort the rest.

In [ ]:
raw_data = {}
failed   = []

for stock in stocks:
    try:
        df = yf.download(stock, interval=INTERVAL, period=PERIOD, progress=False)

        if df.empty:
            print(f'  {stock}: empty response, skipping')
            failed.append(stock)
            continue

        # yfinance >= 0.2 returns MultiIndex columns - flatten them
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.columns = [c.strip().capitalize() for c in df.columns]
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()

        raw_data[stock] = df
        print(f'  {stock}: {len(df):,} candles')

    except Exception as e:
        print(f'  {stock}: failed - {e}')
        failed.append(stock)

print(f'\nDownloaded {len(raw_data)} / {len(stocks)} stocks')
if failed:
    print(f'Skipped: {failed}')

## Step 4 - Simulate One Full Year

We only have two months of real data. To get a fair one-year CAGR we tile
the block six times. Each new block is price-scaled to start exactly where
the previous one ended, so there are no artificial jumps in the series.

In [ ]:
def simulate_one_year(df, iterations=ITERATIONS):
    blocks = []

    for i in range(iterations):
        block = df.copy()

        if i > 0:
            prev_end   = blocks[-1]['Close'].iloc[-1]
            curr_start = block['Close'].iloc[0]
            scale      = prev_end / curr_start
            for col in ['Open', 'High', 'Low', 'Close']:
                block[col] = block[col] * scale

        blocks.append(block)

    return pd.concat(blocks, ignore_index=True)


simulated_data = {}
for stock, df in raw_data.items():
    simulated_data[stock] = simulate_one_year(df)

# Quick check
sample = list(simulated_data.keys())[0]
print(f'Stock           : {sample}')
print(f'Raw candles     : {len(raw_data[sample]):,}')
print(f'Simulated 1yr   : {len(simulated_data[sample]):,} candles')

## Step 5 - Extract Features

Five features per stock are passed to KMeans.

- CAGR: total return over the simulated year
- Volatility: standard deviation of daily returns
- Sharpe ratio: annualised, using 6.5% risk-free rate (India 10Y)
- Momentum: return over the last 20 candles
- Max drawdown: deepest peak-to-trough loss

In [ ]:
def compute_features(df, stock_name):
    close = df['Close'].values.flatten().astype(float)

    start_price = close[0]
    end_price   = close[-1]

    # 1-year CAGR (simulated period is exactly 1 year so exponent = 1)
    cagr = (end_price / start_price) - 1.0

    # Daily returns: compare each day's open to its close
    n_days     = len(close) // CANDLES_PER_DAY
    day_opens  = [close[d * CANDLES_PER_DAY]               for d in range(n_days - 1)]
    day_closes = [close[(d + 1) * CANDLES_PER_DAY - 1]     for d in range(n_days - 1)]
    daily_ret  = np.array([(c - o) / o for o, c in zip(day_opens, day_closes)])

    volatility = float(np.std(daily_ret))

    # Annualised Sharpe ratio
    rf_daily = 0.065 / 252
    excess   = daily_ret - rf_daily
    if np.std(excess) > 0:
        sharpe = float((np.mean(excess) / np.std(excess)) * np.sqrt(252))
    else:
        sharpe = 0.0

    # Momentum: return over the last 20 candles
    momentum = float((close[-1] - close[-20]) / close[-20]) if len(close) > 20 else 0.0

    # Maximum drawdown
    peak   = np.maximum.accumulate(close)
    max_dd = float(((close - peak) / peak).min())

    return {
        'stock'      : stock_name,
        'cagr'       : round(cagr, 6),
        'volatility' : round(volatility, 6),
        'sharpe'     : round(sharpe, 4),
        'momentum'   : round(momentum, 6),
        'max_dd'     : round(max_dd, 6),
        'start_price': round(start_price, 2),
        'end_price'  : round(end_price, 2),
    }


features_list = [compute_features(df, stock) for stock, df in simulated_data.items()]
features_df   = pd.DataFrame(features_list).set_index('stock')

print('Feature table:')
print(features_df.to_string())

## Step 6 - KMeans Clustering

The five features are standardised then KMeans is run five times.
We keep the run with the lowest inertia (tightest, most stable clusters).

In [ ]:
CLUSTER_FEATURES = ['cagr', 'volatility', 'sharpe', 'momentum', 'max_dd']

X        = features_df[CLUSTER_FEATURES].values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

best_kmeans  = None
best_inertia = float('inf')

print(f'Running KMeans {KMEANS_RUNS} times...')
print()

for run in range(KMEANS_RUNS):
    km = KMeans(n_clusters=N_CLUSTERS, random_state=run * 7, n_init=10, max_iter=500)
    km.fit(X_scaled)
    tag = '  <- best' if km.inertia_ < best_inertia else ''
    print(f'  Run {run + 1}:  inertia = {km.inertia_:.4f}{tag}')
    if km.inertia_ < best_inertia:
        best_inertia = km.inertia_
        best_kmeans  = km

features_df['cluster'] = best_kmeans.labels_

print()
print(f'Best inertia: {best_inertia:.4f}')
print()
print('Assignments:')
print(features_df[['cagr', 'cluster']].to_string())

## Step 7 - Find the Outlier

The outlier is the stock that sits farthest from its own cluster centroid.
It is the one that does not quite fit its peer group - interesting to watch.

In [ ]:
centroids = best_kmeans.cluster_centers_

distances = []
for i in range(len(features_df)):
    cid  = int(features_df.iloc[i]['cluster'])
    dist = np.linalg.norm(X_scaled[i] - centroids[cid])
    distances.append(dist)

features_df['dist_to_centroid'] = distances

outlier_stock = features_df['dist_to_centroid'].idxmax()
orow          = features_df.loc[outlier_stock]

print('Outlier stock (farthest from cluster centroid):')
print(f'  Stock    : {outlier_stock}')
print(f'  CAGR     : {orow["cagr"] * 100:.2f}%')
print(f'  Distance : {orow["dist_to_centroid"]:.4f}')
print(f'  Cluster  : {int(orow["cluster"])}')

## Step 8 - Sector Summary Stats

Identify the top and bottom performers and the sector mean before assigning signals.

In [ ]:
highest_cagr_stock = features_df['cagr'].idxmax()
lowest_cagr_stock  = features_df['cagr'].idxmin()
sector_mean_cagr   = features_df['cagr'].mean()

print(f'Highest CAGR : {highest_cagr_stock:<20} {features_df.loc[highest_cagr_stock, "cagr"] * 100:+.2f}%')
print(f'Lowest CAGR  : {lowest_cagr_stock:<20} {features_df.loc[lowest_cagr_stock,  "cagr"] * 100:+.2f}%')
print(f'Sector Mean  : {sector_mean_cagr * 100:+.2f}%')

## Step 9 - Trade Signal Assignment

Each stock gets one signal. The rules check outlier status first,
then top/bottom CAGR, then position relative to sector mean and Sharpe.

In [ ]:
def get_signal(row, sector_mean, outlier, highest, lowest):
    stock  = row.name
    cagr   = row['cagr']
    sharpe = row['sharpe']

    if stock == outlier:
        if cagr > sector_mean:
            return 'STRONG BUY  (Outlier + Above Sector CAGR)'
        else:
            return 'STRONG SELL (Outlier + Below Sector CAGR)'

    if stock == highest:
        return 'BUY         (Highest CAGR in Sector)'

    if stock == lowest:
        return 'AVOID       (Lowest CAGR in Sector)'

    if cagr > sector_mean and sharpe > 0:
        return 'BUY / HOLD  (Above Mean, Positive Sharpe)'

    if cagr > sector_mean and sharpe <= 0:
        return 'HOLD        (Above Mean, Low Sharpe)'

    if cagr <= sector_mean and sharpe > 0:
        return 'NEUTRAL     (Below Mean, Improving Sharpe)'

    return 'AVOID       (Below Mean, Negative Sharpe)'


features_df['signal'] = features_df.apply(
    lambda r: get_signal(r, sector_mean_cagr, outlier_stock,
                         highest_cagr_stock, lowest_cagr_stock),
    axis=1
)

print('Signals assigned to all stocks.')

## Step 10 - Final Report

Stocks sorted by CAGR descending. Outlier marked with an asterisk.

In [ ]:
divider = '=' * 80
line    = '-' * 80

print(divider)
print('  KMEANS SECTOR CAGR ANALYSIS')
print(f'  Sector  : {SECTOR_NAME}')
print(f'  Method  : 5-min candles, 60-day window, tiled x{ITERATIONS} = 1 year simulated')
print(f'  Stocks  : {len(features_df)}    Clusters : {N_CLUSTERS}    KMeans runs : {KMEANS_RUNS}')
print(divider)
print()
print(f'  Sector Mean CAGR : {sector_mean_cagr * 100:+.2f}%')
print(f'  Highest CAGR     : {highest_cagr_stock} ({features_df.loc[highest_cagr_stock, "cagr"] * 100:+.2f}%)')
print(f'  Lowest CAGR      : {lowest_cagr_stock} ({features_df.loc[lowest_cagr_stock, "cagr"] * 100:+.2f}%)')
print(f'  Outlier Stock    : {outlier_stock} (dist = {features_df.loc[outlier_stock, "dist_to_centroid"]:.4f})')
print()
print(line)
print(f'  {"STOCK":<22} {"CAGR":>8}  {"SHARPE":>7}  {"VOLAT":>7}  {"DIST":>7}  SIGNAL')
print(line)

for stock, row in features_df.sort_values('cagr', ascending=False).iterrows():
    mark = '*' if stock == outlier_stock else ' '
    cagr_pct  = row['cagr']       * 100
    volat_pct = row['volatility'] * 100
    print(
        f'  {mark} {stock:<20}  '
        f'{cagr_pct:>+7.2f}%  '
        f'{row["sharpe"]:>7.3f}  '
        f'{volat_pct:>6.2f}%  '
        f'{row["dist_to_centroid"]:>7.4f}  '
        f'{row["signal"]}'
    )

print(line)
print('  * = outlier (farthest from its cluster centroid)')
print(divider)
print()

buy_list  = features_df[features_df['signal'].str.contains('BUY')].index.tolist()
sell_list = features_df[features_df['signal'].str.contains('SELL|AVOID')].index.tolist()
hold_list = features_df[features_df['signal'].str.contains('HOLD|NEUTRAL')].index.tolist()

print('  TRADE SUMMARY')
print(line)
print(f'  BUY  : {", ".join(buy_list)  if buy_list  else "None"}')
print(f'  SELL : {", ".join(sell_list) if sell_list else "None"}')
print(f'  HOLD : {", ".join(hold_list) if hold_list else "None"}')
print()
print('  SECTOR BIAS : ', end='')
if sector_mean_cagr > 0.10:
    print('BULLISH - sector outperforming, lean long')
elif sector_mean_cagr > 0.00:
    print('MILDLY BULLISH - positive but stay selective')
elif sector_mean_cagr > -0.10:
    print('MILDLY BEARISH - under pressure, be cautious')
else:
    print('BEARISH - sector weak, avoid fresh longs')
print(divider)
print('  Note: Simulated data only. Not financial advice.')
print(divider)

## Step 11 - Charts

Three charts: cluster scatter, CAGR bar chart, and a Sharpe vs CAGR scatter.
All use the same colour palette defined at the top.

In [ ]:
# Colours
CLUSTER_COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
POS_COLOR      = '#2E7D32'
NEG_COLOR      = '#C62828'
OUTLIER_COLOR  = '#E53935'
MEAN_COLOR     = '#37474F'

# Prepare labels (strip .NS for cleaner display)
def short(ticker):
    return ticker.replace('.NS', '')

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#FFFFFF')

gs = fig.add_gridspec(2, 2, hspace=0.38, wspace=0.32)
ax_scatter = fig.add_subplot(gs[0, 0])
ax_bar     = fig.add_subplot(gs[0, 1])
ax_sharpe  = fig.add_subplot(gs[1, :])

fig.suptitle(
    f'KMeans Clustering Analysis  -  {SECTOR_NAME}',
    fontsize=16, fontweight='bold', color='#212529', y=1.01
)


# Chart 1: Cluster scatter - CAGR vs Volatility
ax_scatter.set_facecolor('#F8F9FA')
for spine in ax_scatter.spines.values():
    spine.set_edgecolor('#DEE2E6')

for cid in sorted(features_df['cluster'].unique()):
    grp = features_df[features_df['cluster'] == cid]
    ax_scatter.scatter(
        grp['cagr'] * 100,
        grp['volatility'] * 100,
        color=CLUSTER_COLORS[cid % len(CLUSTER_COLORS)],
        s=150, edgecolors='#343A40', linewidths=0.9,
        label=f'Cluster {cid}', zorder=4
    )
    for ticker, row in grp.iterrows():
        is_out = (ticker == outlier_stock)
        ax_scatter.annotate(
            short(ticker),
            xy=(row['cagr'] * 100, row['volatility'] * 100),
            xytext=(5, 4), textcoords='offset points',
            fontsize=7.5,
            color=OUTLIER_COLOR if is_out else '#343A40',
            fontweight='bold' if is_out else 'normal'
        )

ax_scatter.axvline(
    x=sector_mean_cagr * 100,
    color=MEAN_COLOR, linestyle='--', linewidth=1.3, alpha=0.7,
    label=f'Sector mean {sector_mean_cagr * 100:+.1f}%'
)
ax_scatter.set_xlabel('CAGR (%)', fontsize=10)
ax_scatter.set_ylabel('Daily Volatility (%)', fontsize=10)
ax_scatter.set_title('Clusters: CAGR vs Volatility', fontsize=11, fontweight='bold', pad=8)
ax_scatter.legend(fontsize=8, framealpha=0.9, edgecolor='#DEE2E6')


# Chart 2: CAGR bar chart
ax_bar.set_facecolor('#F8F9FA')
for spine in ax_bar.spines.values():
    spine.set_edgecolor('#DEE2E6')

sdf        = features_df.sort_values('cagr', ascending=True)
bar_names  = [short(s) for s in sdf.index]
bar_vals   = sdf['cagr'].values * 100
bar_colors = [POS_COLOR if v >= 0 else NEG_COLOR for v in bar_vals]

bars = ax_bar.barh(
    bar_names, bar_vals,
    color=bar_colors, edgecolor='#495057', linewidth=0.5, height=0.6
)

# Thicker red border on the outlier
out_short = short(outlier_stock)
for bar, name in zip(bars, bar_names):
    if name == out_short:
        bar.set_edgecolor(OUTLIER_COLOR)
        bar.set_linewidth(2.5)

# Value labels
for bar, val in zip(bars, bar_vals):
    ha   = 'left'  if val >= 0 else 'right'
    xpos = val + (0.1 if val >= 0 else -0.1)
    ax_bar.text(
        xpos, bar.get_y() + bar.get_height() / 2,
        f'{val:+.1f}%', va='center', ha=ha,
        fontsize=7.5, fontweight='bold', color='#212529'
    )

ax_bar.axvline(
    x=sector_mean_cagr * 100,
    color=MEAN_COLOR, linestyle='--', linewidth=1.3, alpha=0.8,
    label=f'Sector mean {sector_mean_cagr * 100:+.1f}%'
)

# Mark outlier tick
ax_bar.set_yticks(range(len(bar_names)))
ax_bar.set_yticklabels(
    [f'* {n}' if n == out_short else n for n in bar_names],
    fontsize=8
)
ax_bar.set_xlabel('CAGR (%)', fontsize=10)
ax_bar.set_title('Stock CAGR Comparison  (* = outlier)', fontsize=11, fontweight='bold', pad=8)
ax_bar.legend(fontsize=8, framealpha=0.9, edgecolor='#DEE2E6')
ax_bar.grid(True, color='#E9ECEF', linewidth=0.8, axis='x')


# Chart 3: Sharpe vs CAGR scatter - full width bottom row
ax_sharpe.set_facecolor('#F8F9FA')
for spine in ax_sharpe.spines.values():
    spine.set_edgecolor('#DEE2E6')

for cid in sorted(features_df['cluster'].unique()):
    grp = features_df[features_df['cluster'] == cid]
    ax_sharpe.scatter(
        grp['cagr'] * 100,
        grp['sharpe'],
        color=CLUSTER_COLORS[cid % len(CLUSTER_COLORS)],
        s=140, edgecolors='#343A40', linewidths=0.9,
        label=f'Cluster {cid}', zorder=4
    )
    for ticker, row in grp.iterrows():
        is_out = (ticker == outlier_stock)
        ax_sharpe.annotate(
            short(ticker),
            xy=(row['cagr'] * 100, row['sharpe']),
            xytext=(5, 4), textcoords='offset points',
            fontsize=8,
            color=OUTLIER_COLOR if is_out else '#343A40',
            fontweight='bold' if is_out else 'normal'
        )

ax_sharpe.axhline(y=0, color='#868E96', linestyle='-',  linewidth=0.9, alpha=0.6)
ax_sharpe.axvline(
    x=sector_mean_cagr * 100,
    color=MEAN_COLOR, linestyle='--', linewidth=1.3, alpha=0.7,
    label=f'Sector mean {sector_mean_cagr * 100:+.1f}%'
)
ax_sharpe.set_xlabel('CAGR (%)', fontsize=10)
ax_sharpe.set_ylabel('Sharpe Ratio (annualised)', fontsize=10)
ax_sharpe.set_title('CAGR vs Sharpe Ratio  (top-right quadrant = best risk-adjusted returns)',
                    fontsize=11, fontweight='bold', pad=8)
ax_sharpe.legend(fontsize=8, framealpha=0.9, edgecolor='#DEE2E6')

plt.savefig('sector_kmeans_analysis.png', dpi=150, bbox_inches='tight', facecolor='#FFFFFF')
plt.show()
print('Chart saved as sector_kmeans_analysis.png')

## Step 12 - Export to CSV

Saves the full results table to a CSV file you can open in Excel or Google Sheets.

In [ ]:
out = features_df[['cagr', 'volatility', 'sharpe', 'momentum', 'max_dd',
                    'start_price', 'end_price', 'cluster',
                    'dist_to_centroid', 'signal']].copy()

out.insert(0, 'cagr_pct',       (out['cagr']       * 100).round(2))
out.insert(1, 'volatility_pct', (out['volatility'] * 100).round(2))
out['sector_mean_cagr_pct'] = round(sector_mean_cagr * 100, 2)

out.to_csv('sector_cagr_report.csv')
print('Saved: sector_cagr_report.csv')
print()
print(out[['cagr_pct', 'sharpe', 'cluster', 'dist_to_centroid', 'signal']].to_string())